# Blackjack Game

Plays one hand of blackjack with everything put together:

1. **Waits for a clear table** (no cards in front of me or the dealer).
2. **Bets:** picks a stack from my chip box (`chip_picker` vision + IK) and places it on the bet square.
3. **Waits for the deal**, then reads **my cards** and the **dealer's up card** with the YOLO card model (`card_reader`).
4. **Decides with basic strategy**: **HIT = nod**, **STAND = sway side to side**. Repeats after each new card.
5. If I bust, I lose. Otherwise **waits for the dealer** to flip the hole card and play to 17+.
6. **Settles:** WIN or PUSH = I collect my stack back into my chip box. LOSE = I leave it.

Everything it does gets printed.

**Uses the tuned settings from `chip_picker`** (offsets, grab / place heights, bet square) and the chip box
calibration it saved (`chip_box_calib.json`). If you retune in `chip_picker`, copy the numbers into the settings cell here too.

Run the cells top to bottom, then the **Play** cell. Stop anytime with the stop button.

In [ ]:
%run ../Part1/DofbotKinematics.ipynb

import json
import os
import time
from collections import Counter
import cv2
import matplotlib.pyplot as plt
from Arm_Lib import Arm_Device

arm = Arm_Device()

CAMERA_INDEX = 0   # try 1 if you get no frames
cap = cv2.VideoCapture(CAMERA_INDEX)
time.sleep(1)      # let the camera warm up

with open('poses.json') as f:
    poses = json.load(f)
print("Poses:", list(poses))

## Settings

In [ ]:
# --- Table layout, in the FK base frame (meters). +y = forward at base 90 ---
# The 4 corners of the WHITE square inside the blue tape, measured from the robot base.
# A = corner closest to the robot, then counterclockwise seen from above: B (along +x), C (far corner), D (along +y).
# From the measured drawing: A is 6.5 cm right / 17.5 cm forward; near side 10.3 cm, left side 7.7 cm,
# far side 10.5 cm, right side 7.9 cm (C is the average of the two ways of reaching it).
BOX_CORNERS = {
    'A': (0.065, 0.175),
    'B': (0.168, 0.175),
    'C': (0.169, 0.253),
    'D': (0.065, 0.252),
}

CHIP_DIAMETER = 0.04     # every stack is a 4 cm circle
CHIP_HEIGHT = 0.017      # 1.7 cm, a taped stack we move as one piece

# The 4 corners of the BET square (inner edge of its tape), measured from the robot base like the chip box:
# A = corner closest to the robot, then counterclockwise seen from above. Chips get dropped in the middle.
# From the measured drawing: A is 7.7 in (19.56 cm) left of the base on its centerline, B is 7.5 in (19.05 cm) left
# and 6.4 cm forward; near side 5 cm, far side 4.75 cm, both left/right sides 6.4 cm.
BET_CORNERS = {
    'A': (-0.1956, 0.000),
    'B': (-0.1905, 0.064),
    'C': (-0.2380, 0.064),
    'D': (-0.2456, 0.000),
}

# --- Gripper (servo 6). Find these with camera_pose_tester / the gripper test cell below ---
GRIP_OPEN = 40           # wide enough for a 4 cm chip
GRIP_CLOSED = 135        # holding a chip firmly

# --- Fingers: how far the fingertips reach past the palm (the FK end point), along the gripper ---
# The fingers get LONGER as they close.
FINGER_OPEN = 0.057      # 2.25 in at GRIP_OPEN
FINGER_CLOSED = 0.0635   # 2.5 in at GRIP_CLOSED (closed on a stack)

# --- Fingertip heights (meters above the table). Everything below aims the FINGERTIPS, not the palm ---
HOVER_Z = 0.06           # travel height
GRAB_Z = 0.003           # fingertips once CLOSED on a stack. Use the grab finder (step 5b) to set this
PLACE_Z = 0.006          # fingertips when letting go on the bet square: the stack's bottom ends up ~3 mm above the table

# --- Corrections: the real arm doesn't match FK perfectly (servo offsets, droop) ---
# They're added to every fingertip target. The error depends on where the arm is, so each area has its own set.
# Chip box: found with the grab finder (step 5b)
Z_OFFSET = -0.022        # height. Negative = lower
RADIAL_OFFSET = 0.018    # + = farther from the robot, - = closer
SIDE_OFFSET = -0.003     # + = to the left, - = to the right (standing behind the robot, looking out)
# Bet square: found with the place finder (step 8b)
BET_Z_OFFSET = -0.017
BET_RADIAL_OFFSET = 0.009
BET_SIDE_OFFSET = -0.003

# --- Colors in HSV (OpenCV: H 0-180, S 0-255, V 0-255). Tune with the color test cell ---
COLORS = {
    'blue':  [((95, 80, 40), (130, 255, 255))],     # tape
    'white': [((0, 0, 160), (180, 60, 255))],       # the box floor
    'red':   [((0, 100, 70), (10, 255, 255)), ((170, 100, 70), (180, 255, 255))],  # red wraps around H=0
    'green': [((40, 70, 50), (85, 255, 255))],
    'black': [((0, 0, 0), (180, 255, 60))],
}

CALIB_FILE = 'chip_box_calib.json'
calib = None             # filled in by step 2b

In [ ]:
# --- Game settings ---
BET_COLOR = 'red'        # which chip stack color to bet with
MODEL_PATH = 'models/best.onnx'
CONF = 0.5               # card detections less confident than this are ignored
POLL_S = 2.0             # seconds between looks while waiting on the dealer
SHOW_PICTURES = False    # True = show every camera picture with the card boxes (slower)
GESTURE_POSE = [90, 90, 90, 90, 90]   # arm upright for the hit / stand signals

## Chip vision

In [ ]:
def take_picture():
    global cap
    if not cap.isOpened():
        cap = cv2.VideoCapture(CAMERA_INDEX)
        time.sleep(1)
    # Throw away a few old frames so we get the current view
    for _ in range(5):
        cap.read()
    ok, frame = cap.read()
    return frame if ok else None


def color_mask(frame, color):
    hsv = cv2.cvtColor(cv2.GaussianBlur(frame, (5, 5), 0), cv2.COLOR_BGR2HSV)
    mask = np.zeros(hsv.shape[:2], np.uint8)
    for lo, hi in COLORS[color]:
        mask |= cv2.inRange(hsv, np.array(lo), np.array(hi))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    return mask


def box_world_corners():
    """White square corners A, B, C, D in meters (A = near, C = far). Counterclockwise seen from above."""
    return np.array([BOX_CORNERS[k] for k in 'ABCD'], np.float32)


def inside_box(xy):
    return cv2.pointPolygonTest(box_world_corners().reshape(-1, 1, 2), (float(xy[0]), float(xy[1])), False) >= 0


def line_through(pts):
    """Robust straight line through points (Huber fit ignores bumps). Returns (point, direction)."""
    vx, vy, x0, y0 = cv2.fitLine(pts.astype(np.float32), cv2.DIST_HUBER, 0, 0.01, 0.01).flatten()
    return np.array([x0, y0]), np.array([vx, vy])


def intersect(l1, l2):
    (p1, d1), (p2, d2) = l1, l2
    A = np.array([d1, -d2]).T
    if abs(np.linalg.det(A)) < 1e-9:
        return None
    t = np.linalg.solve(A, p2 - p1)[0]
    return p1 + t * d1


def quad_from_contour(contour):
    """4 corners of a 4-sided outline: rough corners first, then a straight line fitted to each side
    (so chips poking over the tape don't bend the corners), then corners = where the sides cross."""
    hull = cv2.convexHull(contour)
    rough = None
    for eps in (0.01, 0.02, 0.03, 0.05, 0.08):
        approx = cv2.approxPolyDP(hull, eps * cv2.arcLength(hull, True), True)
        if len(approx) == 4:
            rough = approx.reshape(4, 2).astype(np.float32)
            break
    if rough is None:
        return cv2.boxPoints(cv2.minAreaRect(contour)).astype(np.float32)
    pts = contour.reshape(-1, 2).astype(np.float32)
    lines = []
    for i in range(4):
        a, b = rough[i], rough[(i + 1) % 4]
        d = (b - a) / (np.linalg.norm(b - a) + 1e-9)
        rel = pts - a
        along = rel @ d
        off = np.abs(rel @ np.array([-d[1], d[0]]))
        L = np.linalg.norm(b - a)
        near = pts[(off < 6) & (along > 0.1 * L) & (along < 0.9 * L)]   # middle of the side only
        lines.append(line_through(near) if len(near) >= 5 else (a, d))
    corners = [intersect(lines[i - 1], lines[i]) for i in range(4)]
    if any(c is None for c in corners):
        return rough
    return np.array(corners, np.float32)


def find_white_box(frame, min_area=3000):
    """Finds the white square inside the blue tape. Returns its 4 corners (pixels), or None.
    Inside = everything within the tape's outer outline that isn't tape (white floor + any chips).
    Works even if a tall stack hides part of the tape."""
    blue = color_mask(frame, 'blue')
    contours = cv2.findContours(blue, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    if not contours:
        return None
    tape = cv2.convexHull(np.vstack([c for c in contours if cv2.contourArea(c) > 200] or [max(contours, key=cv2.contourArea)]))
    inside = np.zeros(blue.shape, np.uint8)
    cv2.fillPoly(inside, [tape], 255)
    inside &= ~blue
    inside = cv2.morphologyEx(inside, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    contours = cv2.findContours(inside, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[-2]
    contours = [c for c in contours if cv2.contourArea(c) >= min_area]
    if not contours:
        return None
    return quad_from_contour(max(contours, key=cv2.contourArea))


def label_corners(corners_px, cam_xy, frame_shape):
    """Puts the 4 image corners in A, B, C, D order automatically.
    Seen by a camera looking down, the box's counterclockwise order stays counterclockwise in the picture, so only
    4 orders are possible. The right one is where the bottom-middle of the picture lands closest to the camera
    (from FK): the bottom of the picture is always the table nearest the camera."""
    c = np.array(corners_px, np.float32)
    center = c.mean(0)
    c = c[np.argsort(-np.arctan2(c[:, 1] - center[1], c[:, 0] - center[0]))]  # counterclockwise on screen
    h, w = frame_shape[:2]
    bottom = np.array([[[w / 2, h - 1]]], np.float32)
    def score(k):
        H = cv2.getPerspectiveTransform(np.roll(c, -k, axis=0), box_world_corners())
        p = cv2.perspectiveTransform(bottom, H)[0, 0]
        return np.hypot(*(p - np.asarray(cam_xy)))
    best = min(range(4), key=score)
    return np.roll(c, -best, axis=0)


def box_in_view(corners_px, shape, margin=3):
    h, w = shape[:2]
    return bool(np.all((corners_px[:, 0] > margin) & (corners_px[:, 0] < w - margin) &
                       (corners_px[:, 1] > margin) & (corners_px[:, 1] < h - margin)))


def calibrate_from_frame(frame, cam_xyz):
    """White square -> homography (pixel -> table meters). Returns (H, labeled corners) or (None, corners)."""
    corners = find_white_box(frame)
    if corners is None:
        return None, None
    corners = label_corners(corners, cam_xyz[:2], frame.shape)
    if not box_in_view(corners, frame.shape):
        return None, corners
    return cv2.getPerspectiveTransform(corners, box_world_corners()), corners


def px_to_table(H, pts):
    pts = np.array(pts, np.float32).reshape(-1, 1, 2)
    return cv2.perspectiveTransform(pts, H).reshape(-1, 2)


def table_to_px(H, pts):
    return px_to_table(np.linalg.inv(H), pts)


def top_to_table_factor(cam_xyz):
    """The camera sees chip TOPS (CHIP_HEIGHT up). A top point maps onto the table too far from the camera;
    scaling toward the camera by this factor fixes it."""
    return (cam_xyz[2] - CHIP_HEIGHT) / cam_xyz[2]


def fit_circle(pts, radius, cam_xy, res=0.0005, tol=0.003):
    """Finds the center of a 4 cm circle from a stack's outline (table meters, chip-top plane).
    From the camera's angle the FAR half of the top rim is always visible, while the near side is
    smeared by the stack's side wall. So: try every center near the blob and keep the one whose far
    half-circle lands on the outline best.
    Returns (center, coverage) where coverage = fraction of that far half-circle on the outline (1.0 = perfect)."""
    lo = pts.min(0) - radius
    hi = pts.max(0) + radius
    w, h = (np.ceil((hi - lo) / res) + 1).astype(int)
    # Distance (meters) from every grid cell to the nearest outline point
    canvas = np.full((h, w), 255, np.uint8)
    cv2.polylines(canvas, [((pts - lo) / res).round().astype(np.int32)], True, 0, 1)
    dist = cv2.distanceTransform(canvas, cv2.DIST_L2, 3) * res

    away = np.arctan2(*(pts.mean(0) - cam_xy)[::-1])
    t = away + np.linspace(-np.pi / 2, np.pi / 2, 25)
    ring = radius * np.c_[np.cos(t), np.sin(t)]

    def coverage(centers):
        rim = ((centers[:, None, :] + ring[None]) - lo) / res
        u = np.clip(rim[..., 0].round().astype(int), 0, w - 1)
        v = np.clip(rim[..., 1].round().astype(int), 0, h - 1)
        d = dist[v, u]
        return (d < tol).mean(1) - d.mean(1)          # most rim on the outline, then closest

    best = pts.mean(0)
    for span, step in ((0.02, 0.001), (0.0015, 0.00025)):   # coarse grid, then fine
        g = np.arange(-span, span + 1e-9, step)
        cand = best + np.stack(np.meshgrid(g, g), -1).reshape(-1, 2)
        best = cand[coverage(cand).argmax()]
    rim = ((best + ring) - lo) / res
    d = dist[np.clip(rim[:, 1].round().astype(int), 0, h - 1), np.clip(rim[:, 0].round().astype(int), 0, w - 1)]
    return best, float((d < tol).mean())


def box_area_px(H, cam_xyz, shape, grow=0.012):
    """Image area where chip tops in the box can show up (box grown a bit: tall stacks lean outward in the picture)."""
    k = top_to_table_factor(cam_xyz)
    cam = np.array(cam_xyz[:2], float)
    corners = box_world_corners().astype(float)
    out = corners - corners.mean(0)
    top = corners + grow * np.sqrt(2) * out / np.linalg.norm(out, axis=1, keepdims=True)   # push each corner outward
    poly = table_to_px(H, cam + (top - cam) / k)
    area = np.zeros(shape[:2], np.uint8)
    cv2.fillPoly(area, [poly.astype(np.int32)], 255)
    return area


def find_chips(frame, color, H, cam_xyz, min_rim=0.8):
    """Chip stacks of one color inside the box. Each one has to be a 4 cm circle.
    Returns a list of dicts, closest to the robot first."""
    k = top_to_table_factor(cam_xyz)
    cam = np.array(cam_xyz[:2], float)
    R = CHIP_DIAMETER / 2
    mask = color_mask(frame, color) & box_area_px(H, cam_xyz, frame.shape)
    chips = []
    for contour in cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[-2]:
        if cv2.contourArea(contour) < 150:
            continue
        # outline -> table meters, on the plane of the chip tops
        pts = cam + (px_to_table(H, contour[:, 0, :]) - cam) * k
        size = 2 * np.sqrt(cv2.contourArea(pts.astype(np.float32)) / np.pi)
        if not (0.75 * CHIP_DIAMETER <= size <= 1.6 * CHIP_DIAMETER):
            continue                                           # way too small / big for one stack
        center, rim = fit_circle(pts, R, cam)
        if rim < min_rim:
            continue                                           # outline doesn't trace a 4 cm circle
        x, y = center
        if not inside_box((x, y)):
            continue                                           # center outside the white square
        chips.append({'xy': (float(x), float(y)), 'size_cm': size * 100, 'rim': rim})
    chips.sort(key=lambda ch: np.hypot(*ch['xy']))             # closest to the base = easiest to reach
    return chips


def chip_circle_px(H, cam_xyz, xy):
    """The 4 cm circle (top of the stack) drawn back onto the picture."""
    k = top_to_table_factor(cam_xyz)
    cam = np.array(cam_xyz[:2], float)
    t = np.linspace(0, 2 * np.pi, 48)
    top = np.c_[xy[0] + CHIP_DIAMETER / 2 * np.cos(t), xy[1] + CHIP_DIAMETER / 2 * np.sin(t)]
    return table_to_px(H, cam + (top - cam) / k)


def show(frame, chips=(), title='', H=None, corners=None, cam_xyz=None):
    img = frame.copy()
    if corners is not None:
        cv2.polylines(img, [corners.astype(np.int32)], True, (255, 0, 255), 2)
        for (u, v), name in zip(corners, 'ABCD'):
            cv2.putText(img, name, (int(u) + 5, int(v) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 255), 2)
    for i, ch in enumerate(chips):
        ring = chip_circle_px(H, cam_xyz, ch['xy'])
        cv2.polylines(img, [ring.astype(np.int32)], True, (0, 255, 0), 2)
        u, v = ring.mean(0).astype(int)
        cv2.circle(img, (u, v), 4, (0, 0, 255), -1)
        cv2.putText(img, f"#{i} ({ch['xy'][0]*100:.1f},{ch['xy'][1]*100:.1f})cm", (u + 6, v),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 160, 0), 2)
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

## IK

In [ ]:
# Same chain as fk_Dofbot (Part 1), written with numpy arrays so IK can run many guesses at once, fast.
_P01 = (l0 + l1) * ez
_P23 = l2 * ex
_P34 = -l3 * ez
_P5T = -(l4 + l5) * ex


def _rz(t):
    c, s, o, z = np.cos(t), np.sin(t), np.ones_like(t), np.zeros_like(t)
    return np.stack([np.stack([c, -s, z], -1), np.stack([s, c, z], -1), np.stack([z, z, o], -1)], -2)


def _ry(t):
    c, s, o, z = np.cos(t), np.sin(t), np.ones_like(t), np.zeros_like(t)
    return np.stack([np.stack([c, z, s], -1), np.stack([z, o, z], -1), np.stack([-s, z, c], -1)], -2)


def fk_fast(Q):
    """Gripper tip position and pointing direction for joints 1-4 (degrees), for many joint sets at once.
    Same math as fk_Dofbot. Wrist roll (joint 5) only spins the gripper about its own axis,
    so it doesn't change the tip or the pointing direction."""
    t = np.radians(np.atleast_2d(Q))
    R01 = _rz(t[:, 0])
    R02 = R01 @ _ry(-t[:, 1])
    R03 = R02 @ _ry(-t[:, 2])
    R04 = R03 @ _ry(-t[:, 3])
    pos = _P01 + R02 @ _P23 + R03 @ _P34 + R04 @ _P5T
    return pos, -R04[:, :, 0]                       # the gripper points along -x of the tool frame


DOWN = np.array([0, 0, -1.0])
Q_LO, Q_HI = np.zeros(4), np.full(4, 180.0)


def ik_residual(Q, target, w_down, reach):
    pos, pointing = fk_fast(Q)
    tips = pos + reach * pointing                    # fingertips = palm + finger length along the gripper
    return np.concatenate([tips - target, w_down * (pointing - DOWN)], 1)


def ik_Dofbot(target, reach=0.0, w_down=0.003, iters=80):
    """Joint angles [q1..q5] (deg) that put the FINGERTIPS at target (x, y, z in meters). q5 stays 90.
    reach = how far the fingertips stick out past the palm (FINGER_OPEN / FINGER_CLOSED); 0 = the palm itself.
    Position comes first; among good answers it prefers pointing the gripper straight down.
    Same idea as the Jacobian in sensitivity.py (nudge each joint, see how the tip moves, step toward the
    target), run from 12 starting guesses at the same time."""
    target = np.asarray(target, float)
    base = np.degrees(np.arctan2(target[1], target[0]))
    Q = np.array([[base, a, b, c] for a in (10, 40, 80) for b in (60, 100) for c in (0, 40)], float)
    Q = np.clip(Q, Q_LO, Q_HI)
    eye = 1e-6 * np.eye(4)
    for _ in range(iters):
        r = ik_residual(Q, target, w_down, reach)                             # (12, 6)
        J = np.stack([(ik_residual(Q + 0.01 * np.eye(4)[i], target, w_down, reach) - r) / 0.01
                      for i in range(4)], 2)                                  # (12, 6, 4)
        Jt = J.transpose(0, 2, 1)
        step = -np.linalg.solve(Jt @ J + eye, (Jt @ r[..., None]))[..., 0]
        Q = np.clip(Q + np.clip(step, -10, 10), Q_LO, Q_HI)
        if np.abs(step).max() < 1e-3:
            break
    pos, pointing = fk_fast(Q)
    err = np.linalg.norm(pos + reach * pointing - target, axis=1)
    pitch = np.degrees(np.arccos(np.clip(pointing[:, 2], -1, 1)))           # like pitch_from_vertical: 180 = down
    best = np.argmin(err + 0.00005 * (180 - pitch))                         # 1 mm of error ~ 20 deg less tilt
    q_out = [float(round(a, 1)) for a in Q[best]] + [90.0]
    # Double-check the answer with the real Part 1 FK (palm) + finger length
    Rot, Pot = fk_Dofbot(q_out)
    tips = Pot - reach * Rot.as_matrix()[:, 0]       # pointing = -x of the tool frame
    return q_out, float(np.linalg.norm(tips - target)), pitch_from_vertical(Rot)


# Sanity check: fk_fast must agree with fk_Dofbot
for _q in ([90, 90, 90, 90], [60, 117, 0, 0], [45, 30, 120, 60]):
    _R, _P = fk_Dofbot(list(_q) + [90])
    _p, _d = fk_fast(_q)
    assert np.allclose(_p[0], _P, atol=1e-9) and np.allclose(_d[0], -_R.as_matrix()[:, 0], atol=1e-9), "fk_fast != fk_Dofbot"
print("fk_fast matches fk_Dofbot")

## Arm motion, picking and placing chips

In [ ]:
last_q = [90, 90, 90, 90, 90]   # last joint angles we sent (used to open/close without moving)


def move(q, grip, ms=1500):
    # q = [q1..q5] in degrees, grip = servo 6
    global last_q
    last_q = list(q[:5])
    arm.Arm_serial_servo_write6(*q[:5], grip, ms)
    time.sleep(ms / 1000 + 0.5)


def finger_reach(grip):
    """Fingertip distance past the palm for a gripper servo value (straight line between open and closed)."""
    f = np.clip((grip - GRIP_OPEN) / (GRIP_CLOSED - GRIP_OPEN), 0, 1)
    return FINGER_OPEN + f * (FINGER_CLOSED - FINGER_OPEN)


def chip_fix():
    return (Z_OFFSET, RADIAL_OFFSET, SIDE_OFFSET)


def bet_fix():
    return (BET_Z_OFFSET, BET_RADIAL_OFFSET, BET_SIDE_OFFSET)


def corrected(xyz, fix=None):
    """Applies measured corrections (height, radial = toward / away from the base, side). Default: chip-box set."""
    dz, dr, ds = fix if fix is not None else chip_fix()
    x, y, z = xyz
    out = np.array([x, y], float)
    u = out / (np.linalg.norm(out) + 1e-9)          # away from the robot
    left = np.array([-u[1], u[0]])                  # 90 deg counterclockwise = left, looking out from the robot
    out = out + dr * u + ds * left
    return (out[0], out[1], z + dz)


def solve_fingertips(xyz, reach, fix=None, max_err=0.005):
    target = corrected(xyz, fix)
    q, err, pitch = ik_Dofbot(target, reach)
    print(f"  fingertips -> ({xyz[0]*100:.1f}, {xyz[1]*100:.1f}, {xyz[2]*100:.1f}) cm  reach {reach*100:.2f} cm  "
          f"q={q}  err={err*1000:.1f} mm  pitch={pitch:.0f}")
    if err > max_err:
        raise RuntimeError(f"Can't reach that spot (off by {err*1000:.0f} mm).")
    return q


def move_to_xyz(xyz, grip, ms=1500, fix=None):
    """Put the fingertips at xyz, with the gripper at `grip` (its finger length is used)."""
    q = solve_fingertips(xyz, finger_reach(grip), fix)
    move(q, grip, ms)
    return q


def look_for_chips(color, show_it=True):
    """Go to the mychips pose and find chips of one color in the box.
    Uses the saved calibration (taken with the box empty, so no chips hide the tape).
    If there isn't one yet, finds the white square in this picture instead."""
    move(poses['mychips'], GRIP_OPEN)
    frame = take_picture()
    if frame is None:
        raise RuntimeError("No camera frame")
    _, cam = fk_Dofbot(poses['mychips'])          # camera ~ at the gripper
    if calib is not None:
        H, corners = np.array(calib['H']), np.array(calib['corners'], np.float32)
    else:
        H, corners = calibrate_from_frame(frame, cam)
        if H is None:
            show(frame, title='White square not found', corners=corners)
            raise RuntimeError("Can't see the whole white square. Do steps 2a / 2b with the box empty.")
    chips = find_chips(frame, color, H, cam)
    if show_it:
        show(frame, chips, f"{color}: {len(chips)} chip(s)", H, corners, cam)
    return chips


def pick_chip(color):
    chips = look_for_chips(color)
    if not chips:
        print(f"No {color} chips in the box")
        return False
    x, y = chips[0]['xy']
    print(f"Picking {color} chip at ({x*100:.1f}, {y*100:.1f}) cm")
    move_to_xyz((x, y, HOVER_Z), GRIP_OPEN)            # above the chip
    # Plan the grab for the CLOSED finger length: closing makes the fingers longer, so we go down with the
    # gripper open (fingertips ~0.65 cm higher), then closing brings the fingertips down to GRAB_Z.
    q_grab = solve_fingertips((x, y, GRAB_Z), FINGER_CLOSED)
    move(q_grab, GRIP_OPEN, ms=1000)                   # down around it, still open
    move(q_grab, GRIP_CLOSED, ms=800)                  # close (fingers reach GRAB_Z)
    move_to_xyz((x, y, HOVER_Z), GRIP_CLOSED, ms=1000) # lift
    return True


def bet_center():
    if BET_CORNERS is None:
        raise RuntimeError("Set BET_CORNERS (the bet square's 4 corners, in meters) in the settings cell first.")
    pts = np.array([BET_CORNERS[k] for k in 'ABCD'], float)
    return tuple(pts.mean(0))


def place_on_bet(spot=None):
    """Carry the held stack to the bet square (its middle, unless `spot` = (x, y) is given) and set it down."""
    x, y = spot if spot is not None else bet_center()
    fix = bet_fix()
    print(f"Placing at ({x*100:.1f}, {y*100:.1f}) cm")
    move_to_xyz((x, y, HOVER_Z), GRIP_CLOSED, fix=fix)                # carry over, high
    q_place = solve_fingertips((x, y, PLACE_Z), FINGER_CLOSED, fix)
    move(q_place, GRIP_CLOSED, ms=1000)                               # lower the stack
    move(q_place, GRIP_OPEN, ms=800)                                  # let go (fingers shorten as they open)
    move_to_xyz((x, y, HOVER_Z), GRIP_OPEN, ms=1000, fix=fix)         # back up
    move(poses['home'], GRIP_OPEN)

## Card detection

In [ ]:
CLASSES = ['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'DOWN']   # same order as training
IMGSZ = 416        # same size we trained/exported with


def letterbox(frame, size=IMGSZ):
    """Resize keeping the shape, pad the rest with gray (same as YOLO training)."""
    h, w = frame.shape[:2]
    r = min(size / h, size / w)
    nw, nh = int(round(w * r)), int(round(h * r))
    pad_x, pad_y = (size - nw) / 2, (size - nh) / 2
    img = cv2.resize(frame, (nw, nh))
    img = cv2.copyMakeBorder(img, int(round(pad_y - 0.1)), int(round(pad_y + 0.1)),
                             int(round(pad_x - 0.1)), int(round(pad_x + 0.1)),
                             cv2.BORDER_CONSTANT, value=(114, 114, 114))
    return img, r, pad_x, pad_y


def run_model(blob):
    if BACKEND == 'onnxruntime':
        return session.run(None, {input_name: blob})[0]
    net.setInput(blob)
    return net.forward()


def detect(frame, conf=CONF):
    """Returns a list of (label, confidence, (x1, y1, x2, y2)) in the original picture's pixels."""
    img, r, pad_x, pad_y = letterbox(frame)
    blob = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).transpose(2, 0, 1)[None].astype(np.float32) / 255.0
    out = np.asarray(run_model(blob))[0]

    boxes, scores, ids = [], [], []
    if out.ndim == 2 and out.shape[1] == 6:
        # YOLO26 end-to-end output: each row = x1 y1 x2 y2 score class (no NMS needed)
        for x1, y1, x2, y2, s, c in out:
            if s >= conf:
                boxes.append([x1, y1, x2, y2]); scores.append(float(s)); ids.append(int(c))
    else:
        # Older YOLO output: (4 + classes) x N, boxes are cx cy w h -> needs NMS
        if out.shape[0] != 4 + len(CLASSES):
            out = out.T
        cls_scores = out[4:]
        ids_all = cls_scores.argmax(0)
        s_all = cls_scores.max(0)
        keep = s_all >= conf
        cx, cy, w, h = out[:4, keep]
        xywh = np.stack([cx - w / 2, cy - h / 2, w, h], 1)
        s_k, id_k = s_all[keep], ids_all[keep]
        for i in np.array(cv2.dnn.NMSBoxes(xywh.tolist(), s_k.tolist(), conf, 0.5)).flatten():
            x, y, bw, bh = xywh[i]
            boxes.append([x, y, x + bw, y + bh]); scores.append(float(s_k[i])); ids.append(int(id_k[i]))

    results = []
    H, W = frame.shape[:2]
    for (x1, y1, x2, y2), s, c in zip(boxes, scores, ids):
        # undo the letterbox: back to original picture pixels
        x1 = int(np.clip((x1 - pad_x) / r, 0, W)); x2 = int(np.clip((x2 - pad_x) / r, 0, W))
        y1 = int(np.clip((y1 - pad_y) / r, 0, H)); y2 = int(np.clip((y2 - pad_y) / r, 0, H))
        results.append((CLASSES[c], s, (x1, y1, x2, y2)))
    return results


def hand_value(labels):
    """Blackjack total. Aces count 11 unless that busts. DOWN cards are ignored."""
    total, aces = 0, 0
    for l in labels:
        if l == 'DOWN':
            continue
        if l == 'A':
            total += 11; aces += 1
        elif l in ('J', 'Q', 'K', '10'):
            total += 10
        else:
            total += int(l)
    while total > 21 and aces:
        total -= 10; aces -= 1
    return total, aces > 0   # (total, is_soft)

## Load the card model and chip box calibration

In [ ]:
# Chip box calibration from chip_picker step 2b (same folder)
calib = None
if os.path.exists(CALIB_FILE):
    with open(CALIB_FILE) as f:
        calib = json.load(f)
    if calib['pose'] != poses['mychips']:
        print("WARNING: the mychips pose changed since calibrating. Redo chip_picker step 2b.")
print("Chip box calibration:", "loaded" if calib else "none (will find the white square each time)")

# Card model
try:
    import onnxruntime as ort
    session = ort.InferenceSession(MODEL_PATH, providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name
    BACKEND = 'onnxruntime'
except ImportError:
    net = cv2.dnn.readNetFromONNX(MODEL_PATH)
    BACKEND = 'opencv'
print("Card model:", MODEL_PATH, "via", BACKEND)

## Game logic

In [ ]:
RANK_ORDER = {c: i for i, c in enumerate(CLASSES)}


def card_value(label):
    if label == 'A':
        return 11
    if label in ('J', 'Q', 'K', '10'):
        return 10
    return int(label)


def show_cards(frame, dets, title=''):
    img = frame.copy()
    for label, s, (x1, y1, x2, y2) in dets:
        color = (128, 128, 128) if label == 'DOWN' else (0, 255, 0)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, f"{label} {s:.2f}", (x1, max(y1 - 6, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    plt.figure(figsize=(6, 4.5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()


def read_cards(pose_name, tries=4):
    """Go to a camera pose and read the cards there. Takes pictures until two in a row agree
    (so a card being dealt or a bad frame doesn't count). Returns a sorted list of labels."""
    move(poses[pose_name], GRIP_OPEN)
    reads = []
    for _ in range(tries):
        frame = take_picture()
        if frame is None:
            continue
        dets = detect(frame)
        labels = sorted((d[0] for d in dets), key=RANK_ORDER.get)
        if SHOW_PICTURES:
            show_cards(frame, dets, f"{pose_name}: {labels}")
        if reads and labels == reads[-1]:
            return labels
        reads.append(labels)
    return Counter(tuple(r) for r in reads).most_common(1)[0][0] if reads else []


def face_up(labels):
    return [c for c in labels if c != 'DOWN']


def describe(labels):
    total, soft = hand_value(labels)
    cards = ', '.join(labels) if labels else 'no cards'
    return f"{cards} = {'soft ' if soft else ''}{total}"


def basic_strategy(my_cards, dealer_up):
    """Hit or stand from basic strategy (dealer stands on soft 17). Only hit / stand: where the chart
    says double, we hit on hard hands and stand on soft 18. Returns (decision, reason)."""
    total, soft = hand_value(my_cards)
    d = card_value(dealer_up)
    if soft:
        if total >= 19:
            return 'STAND', f"soft {total} is strong"
        if total == 18:
            return ('STAND', "soft 18 stands against 2-8") if d <= 8 else ('HIT', "soft 18 hits against 9, 10, A")
        return 'HIT', f"soft {total} can't bust, always hit"
    if total >= 17:
        return 'STAND', f"hard {total}: always stand"
    if total >= 13:
        return ('STAND', f"hard {total} vs a weak dealer card (2-6): let the dealer bust") if d <= 6 else \
               ('HIT', f"hard {total} vs a strong dealer card (7-A): hit")
    if total == 12:
        return ('STAND', "hard 12 vs 4-6: stand") if 4 <= d <= 6 else ('HIT', "hard 12 vs 2, 3, 7-A: hit")
    return 'HIT', f"hard {total} can't bust, always hit"


# --- Signals -----------------------------------------------------------------
def signal_hit():
    """Nod: tip the wrist forward and back twice."""
    move(GESTURE_POSE, GRIP_OPEN, ms=1000)
    for _ in range(2):
        move(GESTURE_POSE[:3] + [50, 90], GRIP_OPEN, ms=400)
        move(GESTURE_POSE[:3] + [130, 90], GRIP_OPEN, ms=400)
    move(GESTURE_POSE, GRIP_OPEN, ms=400)


def signal_stand():
    """Wave off: sway the whole arm side to side twice."""
    move(GESTURE_POSE, GRIP_OPEN, ms=1000)
    for _ in range(2):
        move([60] + GESTURE_POSE[1:], GRIP_OPEN, ms=500)
        move([120] + GESTURE_POSE[1:], GRIP_OPEN, ms=500)
    move(GESTURE_POSE, GRIP_OPEN, ms=500)


# --- Chips -------------------------------------------------------------------
def place_at(x, y, fix):
    """Set the held stack down at (x, y) and let go."""
    move_to_xyz((x, y, HOVER_Z), GRIP_CLOSED, fix=fix)
    q = solve_fingertips((x, y, PLACE_Z), FINGER_CLOSED, fix)
    move(q, GRIP_CLOSED, ms=1000)
    move(q, GRIP_OPEN, ms=800)
    move_to_xyz((x, y, HOVER_Z), GRIP_OPEN, ms=1000, fix=fix)


def pick_at(x, y, fix):
    """Grab the stack at (x, y) and lift it."""
    move_to_xyz((x, y, HOVER_Z), GRIP_OPEN, fix=fix)
    q = solve_fingertips((x, y, GRAB_Z), FINGER_CLOSED, fix)
    move(q, GRIP_OPEN, ms=1000)
    move(q, GRIP_CLOSED, ms=800)
    move_to_xyz((x, y, HOVER_Z), GRIP_CLOSED, ms=1000, fix=fix)


def free_spot_in_box():
    """Look at the chip box and return the open spot (x, y) farthest from every stack already there."""
    stacks = [ch['xy'] for color in ('red', 'green', 'black') for ch in look_for_chips(color, show_it=SHOW_PICTURES)]
    corners = box_world_corners().astype(float)
    best, best_gap = None, -1
    for u in np.linspace(0.2, 0.8, 7):
        for v in np.linspace(0.25, 0.75, 5):
            p = (1-u)*(1-v)*corners[0] + u*(1-v)*corners[1] + u*v*corners[2] + (1-u)*v*corners[3]
            gap = min([np.hypot(*(p - np.array(s))) for s in stacks] + [1.0])
            if gap > best_gap:
                best, best_gap = p, gap
    if best_gap < CHIP_DIAMETER:
        print(f"   (chip box is crowded: closest stack will be {best_gap*100:.1f} cm away)")
    return tuple(best)


def place_bet():
    print(f"BET: picking a {BET_COLOR} stack from my chip box...")
    if not pick_chip(BET_COLOR):
        return False
    x, y = bet_center()
    print("BET: putting it on the bet square...")
    place_at(x, y, bet_fix())
    move(poses['home'], GRIP_OPEN)
    print("BET: placed.")
    return True


def collect_bet():
    spot = free_spot_in_box()
    print(f"COLLECT: taking my stack off the bet square -> chip box at ({spot[0]*100:.1f}, {spot[1]*100:.1f}) cm")
    x, y = bet_center()
    pick_at(x, y, bet_fix())
    place_at(spot[0], spot[1], chip_fix())
    move(poses['home'], GRIP_OPEN)
    print("COLLECT: done.")


# --- Waiting on the dealer -----------------------------------------------------
def wait_until(check, waiting_msg, poll=None):
    """Calls check() until it returns something truthy. Prints the waiting message once."""
    said = False
    while True:
        result = check()
        if result:
            return result
        if not said:
            print(waiting_msg)
            said = True
        time.sleep(POLL_S if poll is None else poll)


# --- One hand ------------------------------------------------------------------
def play_hand():
    print("\n================ NEW HAND ================")

    # 1. Table has to be clear before betting
    def table_check():
        mine, dealer = read_cards('mycards'), read_cards('dealer')
        if not mine and not dealer:
            return True
        print(f"   table not clear yet: mine [{', '.join(mine)}], dealer [{', '.join(dealer)}]")
        return False
    wait_until(table_check, "WAIT: clear the table so I can bet...")
    print("TABLE: clear, no cards out.")

    # 2. Bet
    if not place_bet():
        print(f"GAME OVER: no {BET_COLOR} chips left to bet.")
        return 'NO CHIPS'

    # 3. Wait for the deal: 2 cards for me, an up card for the dealer
    def dealt():
        mine, dealer = read_cards('mycards'), read_cards('dealer')
        if len(face_up(mine)) >= 2 and len(face_up(dealer)) >= 1:
            return mine, dealer
        return None
    mine, dealer = wait_until(dealt, "WAIT: dealer, deal the cards (2 for me, 1 up card for you)...")
    up = face_up(dealer)[0]
    print(f"DEALT: me [{describe(mine)}] | dealer up card {up}" + (" + 1 face down" if 'DOWN' in dealer else ""))

    # 4. My turn
    while True:
        total, soft = hand_value(mine)
        if total > 21:
            print(f"BUST: {describe(mine)}.")
            break
        if total == 21:
            print("BLACKJACK!" if len(mine) == 2 else "21!")
            print("STAND (signal: sway)")
            signal_stand()
            break
        decision, reason = basic_strategy(mine, up)
        print(f"DECIDE: me {describe(mine)} vs dealer {up} -> {decision} ({reason})")
        if decision == 'STAND':
            print("STAND (signal: sway)")
            signal_stand()
            break
        print("HIT (signal: nod)")
        signal_hit()
        n_before = len(mine)
        def new_card():
            cards = read_cards('mycards')
            return cards if len(cards) > n_before else None
        mine = wait_until(new_card, "WAIT: dealer, give me a card...")
        print(f"NEW CARD: me [{describe(mine)}]")

    my_total, _ = hand_value(mine)
    if my_total > 21:
        print("RESULT: LOSE (bust). The dealer keeps my bet.")
        move(poses['home'], GRIP_OPEN)
        return 'LOSE'

    # 5. Dealer's turn: wait until the hole card is flipped and the dealer has 17+ (or busts)
    last = [None]
    def dealer_done():
        dealer = read_cards('dealer')
        if dealer != last[0]:
            print(f"   dealer shows [{describe(face_up(dealer))}]" + (" + face down" if 'DOWN' in dealer else ""))
            last[0] = dealer
        total, _ = hand_value(dealer)
        if 'DOWN' not in dealer and len(dealer) >= 2 and total >= 17:
            return dealer
        return None
    dealer = wait_until(dealer_done, "WAIT: dealer, play out your hand (flip the hole card, hit to 17)...")
    dealer_total, _ = hand_value(dealer)
    print(f"DEALER: [{describe(dealer)}]")

    # 6. Settle
    if dealer_total > 21:
        result = 'WIN'
        print(f"RESULT: WIN, dealer busts with {dealer_total}.")
    elif my_total > dealer_total:
        result = 'WIN'
        print(f"RESULT: WIN, {my_total} beats {dealer_total}.")
    elif my_total == dealer_total:
        result = 'PUSH'
        print(f"RESULT: PUSH, both {my_total}. Taking my bet back.")
    else:
        result = 'LOSE'
        print(f"RESULT: LOSE, {dealer_total} beats {my_total}. The dealer keeps my bet.")

    if result in ('WIN', 'PUSH'):
        collect_bet()
    move(poses['home'], GRIP_OPEN)
    return result

## Check before playing (no chips move)
Reads both card areas once and shows the pictures, and tests the two signals.

In [ ]:
SHOW_PICTURES = True
print("My cards:", describe(read_cards('mycards')))
print("Dealer:  ", describe(read_cards('dealer')))
SHOW_PICTURES = False

print("Signal: HIT (nod)")
signal_hit()
print("Signal: STAND (sway)")
signal_stand()
move(poses['home'], GRIP_OPEN)

## Basic strategy table (what I'll do with each hand)

In [ ]:
ups = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'A']
print('dealer up:     ' + ' '.join(f'{u:>2}' for u in ups))
for name, hand in [(f'hard {t}', h) for t, h in [(8, ['3', '5']), (11, ['5', '6']), (12, ['10', '2']), (13, ['10', '3']),
                                               (16, ['10', '6']), (17, ['10', '7'])]] + \
                  [(f'soft {t}', h) for t, h in [(13, ['A', '2']), (17, ['A', '6']), (18, ['A', '7']), (19, ['A', '8'])]]:
    print(f'{name:>8} {"+".join(hand):>6}: ' + ' '.join(f'{basic_strategy(hand, u)[0][0]:>2}' for u in ups))
print('H = hit, S = stand')

## Play a hand

In [ ]:
result = play_hand()
print("\nHAND OVER:", result)

## Done

In [ ]:
move(poses['home'], GRIP_OPEN)
cap.release()